# Simple RAG Pipeline

A minimal, self-contained **Retrieval-Augmented Generation (RAG)** pipeline you can run end-to-end in a notebook.

**Flow:** `corpus -> chunk -> embed -> store -> retrieve (cosine similarity) -> generate answer`

**Design notes**
- **Embeddings:** uses [`sentence-transformers`](https://www.sbert.net/) (`all-MiniLM-L6-v2`) for semantic search. If it isn't installed, it automatically falls back to a TF-IDF embedder (`scikit-learn`) so the notebook still runs.
- **Generation:** uses the OpenAI API if `OPENAI_API_KEY` is set, otherwise falls back to a simple **extractive** answer built from the retrieved chunks. So this runs with **no API key** required.
- No vector DB needed — the "index" is just a NumPy matrix with cosine similarity. Swap in FAISS / Chroma / pgvector later if you need scale.

**Optional install**
```bash
pip install numpy scikit-learn sentence-transformers openai
```


In [1]:
# --- Config ---
from dataclasses import dataclass

@dataclass
class Config:
    embed_model: str = "all-MiniLM-L6-v2"  # sentence-transformers model
    chunk_size: int = 60                   # words per chunk
    chunk_overlap: int = 15                # words overlapping between chunks
    top_k: int = 3                         # chunks to retrieve per query
    openai_model: str = "gpt-4o-mini"      # used only if OPENAI_API_KEY is set

cfg = Config()
cfg


Config(embed_model='all-MiniLM-L6-v2', chunk_size=60, chunk_overlap=15, top_k=3, openai_model='gpt-4o-mini')

## 1. Corpus

A tiny knowledge base. Replace these strings with your own documents (or load from files / a database).

In [2]:
documents = [
    {
        "id": "doc1",
        "title": "What is RAG",
        "text": (
            "Retrieval-Augmented Generation (RAG) is a technique that combines a "
            "retrieval system with a generative language model. Instead of relying "
            "only on knowledge baked into the model's weights, RAG fetches relevant "
            "documents from an external knowledge base and passes them to the model "
            "as context. This reduces hallucinations and lets the model answer "
            "questions about private or up-to-date data."
        ),
    },
    {
        "id": "doc2",
        "title": "Embeddings",
        "text": (
            "An embedding is a dense vector of floating point numbers that represents "
            "the meaning of a piece of text. Texts with similar meaning have vectors "
            "that are close together. Sentence-transformers models like "
            "all-MiniLM-L6-v2 map sentences into a 384-dimensional space. We compare "
            "vectors using cosine similarity to find semantically related text."
        ),
    },
    {
        "id": "doc3",
        "title": "Chunking",
        "text": (
            "Long documents are split into smaller chunks before embedding. Chunking "
            "keeps each vector focused on a single idea and ensures chunks fit within "
            "the model context window. A common strategy uses a fixed window of words "
            "with some overlap so that information spanning a boundary is not lost."
        ),
    },
    {
        "id": "doc4",
        "title": "Vector search",
        "text": (
            "At query time the user's question is embedded with the same model used "
            "for the documents. The query vector is compared against all chunk vectors "
            "and the top-k most similar chunks are retrieved. For small corpora a brute "
            "force NumPy dot product is enough; for millions of vectors use an "
            "approximate nearest neighbor index such as FAISS, HNSW, or a vector "
            "database like Chroma or pgvector."
        ),
    },
]

print(f"{len(documents)} documents loaded")


4 documents loaded


## 2. Chunking

Split each document into overlapping word windows.

In [3]:
def chunk_text(text, chunk_size, overlap):
    words = text.split()
    if not words:
        return []
    step = max(1, chunk_size - overlap)
    chunks = []
    for start in range(0, len(words), step):
        window = words[start:start + chunk_size]
        if window:
            chunks.append(" ".join(window))
        if start + chunk_size >= len(words):
            break
    return chunks


# Build a flat list of chunks, keeping a reference back to the source document.
chunks = []
for doc in documents:
    for i, ch in enumerate(chunk_text(doc["text"], cfg.chunk_size, cfg.chunk_overlap)):
        chunks.append({
            "doc_id": doc["id"],
            "title": doc["title"],
            "chunk_id": f"{doc['id']}::{i}",
            "text": ch,
        })

print(f"{len(chunks)} chunks created")
print("Example chunk:\n", chunks[0]["text"][:200], "...")


5 chunks created
Example chunk:
 Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a generative language model. Instead of relying only on knowledge baked into the model's weights, RAG fetches  ...


## 3. Embedder

Try `sentence-transformers` for real semantic embeddings; fall back to TF-IDF if it's unavailable. Both expose the same `.encode(texts)` interface returning **L2-normalized** vectors, so cosine similarity is just a dot product.

In [4]:
import numpy as np


def _l2_normalize(matrix):
    matrix = np.asarray(matrix, dtype="float32")
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return matrix / norms


class SentenceTransformerEmbedder:
    def __init__(self, model_name):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name)
        self.name = f"sentence-transformers:{model_name}"

    def encode(self, texts):
        vecs = self.model.encode(list(texts), show_progress_bar=False)
        return _l2_normalize(vecs)


class TfidfEmbedder:
    """Lightweight fallback: fit a TF-IDF vocabulary on the corpus."""
    def __init__(self, corpus):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self.vectorizer.fit(corpus)
        self.name = "tfidf:sklearn"

    def encode(self, texts):
        vecs = self.vectorizer.transform(list(texts)).toarray()
        return _l2_normalize(vecs)


def build_embedder(cfg, corpus):
    try:
        emb = SentenceTransformerEmbedder(cfg.embed_model)
        print(f"Using semantic embedder: {emb.name}")
        return emb
    except Exception as e:  # noqa: BLE001
        print(f"sentence-transformers unavailable ({e.__class__.__name__}); using TF-IDF fallback")
        return TfidfEmbedder(corpus)


embedder = build_embedder(cfg, [c["text"] for c in chunks])


sentence-transformers unavailable (ModuleNotFoundError); using TF-IDF fallback


## 4. Build the index

Embed every chunk once and stack the vectors into a matrix. This matrix *is* our vector store.

In [5]:
chunk_texts = [c["text"] for c in chunks]
chunk_matrix = embedder.encode(chunk_texts)  # shape: (n_chunks, dim), L2-normalized
print("Index shape:", chunk_matrix.shape)


Index shape: (5, 110)


## 5. Retrieval

Embed the query, compute cosine similarity against all chunks, return the top-k.

In [6]:
def retrieve(query, k=None):
    k = k or cfg.top_k
    q_vec = embedder.encode([query])[0]           # (dim,)
    scores = chunk_matrix @ q_vec                  # cosine sim (vectors are normalized)
    top_idx = np.argsort(scores)[::-1][:k]
    results = []
    for rank, idx in enumerate(top_idx):
        results.append({
            "rank": rank + 1,
            "score": float(scores[idx]),
            **chunks[idx],
        })
    return results


# quick sanity check
for r in retrieve("How does the system find relevant text?"):
    print(f"[{r['rank']}] score={r['score']:.3f} | {r['title']} | {r['text'][:90]}...")


[1] score=0.232 | Embeddings | An embedding is a dense vector of floating point numbers that represents the meaning of a ...
[2] score=0.107 | What is RAG | Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with ...
[3] score=0.000 | Vector search | millions of vectors use an approximate nearest neighbor index such as FAISS, HNSW, or a ve...


## 6. Generation

Build a grounded prompt from the retrieved context. Use OpenAI if `OPENAI_API_KEY` is available; otherwise return a clean extractive answer so the notebook always produces output.

In [7]:
import os
import textwrap


def build_prompt(query, contexts):
    context_block = "\n\n".join(
        f"[{i+1}] (source: {c['title']})\n{c['text']}" for i, c in enumerate(contexts)
    )
    return (
        "You are a helpful assistant. Answer the QUESTION using ONLY the CONTEXT "
        "below. If the context does not contain the answer, say you don't know. "
        "Cite sources as [number].\n\n"
        f"CONTEXT:\n{context_block}\n\n"
        f"QUESTION: {query}\n\nANSWER:"
    )


def generate(query, contexts):
    prompt = build_prompt(query, contexts)
    api_key = os.environ.get("OPENAI_API_KEY")
    if api_key:
        try:
            from openai import OpenAI
            client = OpenAI()
            resp = client.chat.completions.create(
                model=cfg.openai_model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
            )
            return resp.choices[0].message.content.strip(), "openai"
        except Exception as e:  # noqa: BLE001
            print(f"OpenAI call failed ({e}); using extractive fallback")

    # --- Extractive fallback (no LLM) ---
    answer = "Based on the retrieved context:\n"
    for i, c in enumerate(contexts):
        snippet = textwrap.shorten(c["text"], width=200, placeholder="...")
        answer += f"\n[{i+1}] {snippet}"
    return answer, "extractive-fallback"


## 7. The full RAG pipeline

`ask()` ties retrieval + generation together.

In [8]:
def ask(query, k=None, show_sources=True):
    contexts = retrieve(query, k=k)
    answer, mode = generate(query, contexts)
    print(f"Q: {query}")
    print(f"\nA ({mode}):\n{answer}")
    if show_sources:
        print("\nSources:")
        for c in contexts:
            print(f"  - [{c['rank']}] {c['title']} (score={c['score']:.3f})")
    print("=" * 80)
    return answer


_ = ask("What is retrieval-augmented generation and why is it useful?")
_ = ask("Why do we split documents into chunks?")
_ = ask("What should I use for vector search at large scale?")


Q: What is retrieval-augmented generation and why is it useful?

A (extractive-fallback):
Based on the retrieved context:

[1] Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a generative language model. Instead of relying only on knowledge baked into the model's weights, RAG...
[2] millions of vectors use an approximate nearest neighbor index such as FAISS, HNSW, or a vector database like Chroma or pgvector.
[3] At query time the user's question is embedded with the same model used for the documents. The query vector is compared against all chunk vectors and the top-k most similar chunks are retrieved. For...

Sources:
  - [1] What is RAG (score=0.351)
  - [2] Vector search (score=0.000)
  - [3] Vector search (score=0.000)
Q: Why do we split documents into chunks?

A (extractive-fallback):
Based on the retrieved context:

[1] Long documents are split into smaller chunks before embedding. Chunking keeps each vector focused on a single idea and 

## Next steps

- **Bigger corpus:** load from files, PDFs, or a database instead of the inline list.
- **Real vector store:** swap the NumPy matrix for FAISS, Chroma, or pgvector.
- **Better generation:** set `OPENAI_API_KEY` (or plug in any LLM) to get fluent, cited answers.
- **Evaluation:** add metrics like retrieval hit-rate and answer faithfulness.
- **Metadata filtering:** filter chunks by `doc_id`/tags before ranking.
